# The Model Arena: Benchmarking Frontier AI on Complex Logic

## Architect's Focus: Beyond the Hype

Every week, a new model is released claiming to be "the best." As an Architect, you cannot rely on marketing benchmarks. You must build your own **Evaluation Pipeline**.

In this lecture, we apply the **Universal Connector** pattern to build a "Battlefield." We will send identical logic puzzles to OpenAI, DeepSeek, and xAI (Grok) simultaneously. We will then evaluate them on three critical metrics:
1. **Accuracy:** Did it solve the logic trap?
2. **Latency:** How long did it take to think?
3. **Cost-Efficiency:** Was the result worth the price?

In [1]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# Load environment variables
load_dotenv(override=True)

# 1. Initialize our "Frontier" Clients
client_openai = OpenAI() # GPT-5 Series
client_deepseek = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
client_xai = OpenAI(api_key=os.getenv("XAI_API_KEY"), base_url="https://api.x.ai/v1")

print("Arena Initialized: OpenAI vs. DeepSeek vs. xAI (Grok)")

Arena Initialized: OpenAI vs. DeepSeek vs. xAI (Grok)


## 1. The Benchmarking Orchestrator

We create a single function that handles the query, tracks the time (latency), and captures the response.

In [2]:
def arena_battle(client, model_name, provider_label, prompt):
    start_time = time.time()
    try:
        # Note: We use the universal Chat Completion pattern
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        end_time = time.time()
        
        # Extract text
        answer = response.choices[0].message.content
        latency = end_time - start_time
        
        return {
            "provider": provider_label,
            "answer": answer,
            "latency": latency,
            "tokens": response.usage.total_tokens
        }
    except Exception as e:
        return {"provider": provider_label, "error": str(e)}

print("Orchestrator Function Ready")

Orchestrator Function Ready


## 2. The Logic Gauntlet

In [3]:
puzzles = [
    ("Logic Riddle", "There are three boxes: A, B, and C. Box A contains a red ball. I move the ball to Box B. Then, I swap the physical positions of Box A and Box C. Finally, I take the ball out of its current box and put it in the box that is currently in the middle. Where is the red ball?"),
    ("Linguistic Trap", "How many 'r's are in the word 'strawberry'?"),
    ("Bayesian Math", "A disease affects 1% of the population. A test for it is 95% accurate. If I test positive, what is the % chance I actually have it?")
]

models_to_test = [
    (client_openai, "gpt-5", "OpenAI (Intuitive)"),
    (client_deepseek, "deepseek-reasoner", "DeepSeek (Reasoning)"),
    (client_xai, "grok-4.20-0309-reasoning", "xAI (Frontier)")
]

for title, prompt in puzzles:
    display(Markdown(f"## 🚩 Test: {title}"))
    display(Markdown(f"**Prompt:** {prompt}"))
    
    for client, model, label in models_to_test:
        result = arena_battle(client, model, label, prompt)
        
        if "error" in result:
            print(f"❌ {label} Error: {result['error']}")
            continue
            
        display(Markdown(f"### 🤖 {result['provider']}"))
        display(Markdown(result['answer']))
        print(f"⏱️ Latency: {result['latency']:.2f}s | 🪙 Tokens: {result['tokens']}")
        print("-" * 30)

## 🚩 Test: Logic Riddle

**Prompt:** There are three boxes: A, B, and C. Box A contains a red ball. I move the ball to Box B. Then, I swap the physical positions of Box A and Box C. Finally, I take the ball out of its current box and put it in the box that is currently in the middle. Where is the red ball?

### 🤖 OpenAI (Intuitive)

Box B.

Reason: After moving the ball to B, swapping A and C doesn’t move B (the middle box stays the same). Putting the ball into the “box that is currently in the middle” just puts it back into B.

⏱️ Latency: 45.50s | 🪙 Tokens: 1863
------------------------------


...

The probability that you actually have the disease given a positive test result is approximately 16.1%. This is calculated using Bayes' theorem, considering a 1% disease prevalence and a test that is 95% accurate (with 95% sensitivity and 95% specificity).

⏱️ Latency: 20.16s | 🪙 Tokens: 909
------------------------------


### 🤖 xAI (Frontier)

**~16.1%**

### Quick Explanation (Bayes' theorem)

- Disease prevalence: 1% (so 99% do *not* have it)
- Test is 95% accurate → 95% true positive rate and 5% false positive rate

Probability you actually have the disease given a positive test:

$$
P(\text{Disease} \mid \text{Positive}) = \frac{0.95 \times 0.01}{0.95 \times 0.01 + 0.05 \times 0.99} = \frac{0.0095}{0.059} \approx 0.161 \ (16.1\%)
$$

### Intuitive way (10,000 people)

- 100 people have the disease → test correctly identifies 95 of them
- 9,900 people don't → test gives 5% false positives = 495 wrong positives
- Total positive tests: 95 + 495 = 590
- Of those 590 positive results, only 95 are true cases → **95/590 ≈ 16.1%**

This is the classic "base rate fallacy" result — even with a pretty good test, when the disease is rare, most positive results are false positives.

⏱️ Latency: 4.75s | 🪙 Tokens: 1029
------------------------------
